In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.functions import when
from pyspark.sql.functions import count

# ==========================================
# Create Spark Session
# ==========================================

spark = (
    SparkSession.builder
    .appName("Gaming Analytics - Data Exploration")
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .config( "spark.local.dir", "D:/spark_temp")
    .config( "spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("=" * 60)
print("Spark Version :", spark.version)
print("=" * 60)

Spark Version : 3.5.1


In [11]:
# ==========================================
# Read Datasets
# ==========================================

games = spark.read.csv(
    "../data/bronze/steam_games.csv",
    header=True,
    inferSchema=True
)

reviews = spark.read.csv(
    "../data/bronze/steam_reviews.csv",
    header=True,
    inferSchema=True
)

In [12]:
print("\n========== DATASET OVERVIEW ==========\n")

print("Steam Games")
print(f"Rows    : {games.count()}")
print(f"Columns : {len(games.columns)}")

print("\nSteam Reviews")
print(f"Rows    : {reviews.count()}")
print(f"Columns : {len(reviews.columns)}")

print("\n========== GAMES SCHEMA ==========\n")

games.printSchema()

print("\n========== REVIEWS SCHEMA ==========\n")

reviews.printSchema()


========== DATASET OVERVIEW ==========

Steam Games
Rows    : 27075
Columns : 18

Steam Reviews
Rows    : 6417106
Columns : 5

========== GAMES SCHEMA ==========

root
 |-- appid: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- english: string (nullable = true)
 |-- developer: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- platforms: string (nullable = true)
 |-- required_age: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- steamspy_tags: string (nullable = true)
 |-- achievements: string (nullable = true)
 |-- positive_ratings: string (nullable = true)
 |-- negative_ratings: integer (nullable = true)
 |-- average_playtime: integer (nullable = true)
 |-- median_playtime: integer (nullable = true)
 |-- owners: string (nullable = true)
 |-- price: string (nullable = true)


========== REVIEWS SCHEMA ==========

root
 |-- app_id: integer (nu

In [13]:
print("\n========== SAMPLE GAMES ==========\n")

games.show(5, truncate=False)

print("\n========== SAMPLE REVIEWS ==========\n")

reviews.show(5, truncate=False)


========== SAMPLE GAMES ==========

+-----+-------------------------+------------+-------+----------------+---------+-----------------+------------+----------------------------------------------------------------------------+------+----------------------------+------------+----------------+----------------+----------------+---------------+-----------------+-----+
|appid|name                     |release_date|english|developer       |publisher|platforms        |required_age|categories                                                                  |genres|steamspy_tags               |achievements|positive_ratings|negative_ratings|average_playtime|median_playtime|owners           |price|
+-----+-------------------------+------------+-------+----------------+---------+-----------------+------------+----------------------------------------------------------------------------+------+----------------------------+------------+----------------+----------------+----------------+--------------

In [14]:
print("\n========== GAMES COLUMNS ==========\n")

print(games.columns)

print("\n========== REVIEWS COLUMNS ==========\n")

print(reviews.columns)


========== GAMES COLUMNS ==========

['appid', 'name', 'release_date', 'english', 'developer', 'publisher', 'platforms', 'required_age', 'categories', 'genres', 'steamspy_tags', 'achievements', 'positive_ratings', 'negative_ratings', 'average_playtime', 'median_playtime', 'owners', 'price']

========== REVIEWS COLUMNS ==========

['app_id', 'app_name', 'review_text', 'review_score', 'review_votes']


In [15]:
print("\n========== GAMES STATISTICS ==========\n")

games.describe().show()

print("\n========== REVIEWS STATISTICS ==========\n")

reviews.describe().show()



========== GAMES STATISTICS ==========

+-------+------------------+---------------+--------------------+-------------------+--------------------+-----------------+-----------------+------------------+-----------------+--------------+--------------------+------------------+------------------+------------------+-----------------+------------------+-----------------+------------------+
|summary|             appid|           name|        release_date|            english|           developer|        publisher|        platforms|      required_age|       categories|        genres|       steamspy_tags|      achievements|  positive_ratings|  negative_ratings| average_playtime|   median_playtime|           owners|             price|
+-------+------------------+---------------+--------------------+-------------------+--------------------+-----------------+-----------------+------------------+-----------------+--------------+--------------------+------------------+------------------+------------

In [16]:
print("\n========== MISSING VALUES : GAMES ==========\n")

games.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in games.columns
]).show(vertical=True)


========== MISSING VALUES : GAMES ==========

-RECORD 0---------------
 appid            | 0   
 name             | 0   
 release_date     | 0   
 english          | 0   
 developer        | 0   
 publisher        | 0   
 platforms        | 0   
 required_age     | 0   
 categories       | 0   
 genres           | 0   
 steamspy_tags    | 0   
 achievements     | 0   
 positive_ratings | 0   
 negative_ratings | 0   
 average_playtime | 0   
 median_playtime  | 0   
 owners           | 0   
 price            | 0   



In [17]:
print("\n========== MISSING VALUES : REVIEWS ==========\n")

reviews.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in reviews.columns
]).show(vertical=True)

# print("\n========== DUPLICATES ==========\n")

# print(
#     "Games duplicates :",
#     games.count() -
#     games.dropDuplicates().count()
# )

# print(
#     "Reviews duplicates :",
#     reviews.count() -
#     reviews.dropDuplicates().count()
# )

print("\n========== DUPLICATE CHECK ==========\n")

print("Skipping full duplicate check due to dataset size.")
print("Will perform targeted duplicate checks during cleaning phase.")


========== MISSING VALUES : REVIEWS ==========

-RECORD 0--------------
 app_id       | 0      
 app_name     | 183234 
 review_text  | 7305   
 review_score | 289    
 review_votes | 289    


========== DUPLICATE CHECK ==========

Skipping full duplicate check due to dataset size.
Will perform targeted duplicate checks during cleaning phase.


In [18]:
print("\n========== POSSIBLE JOIN KEYS ==========\n")

print("Games")

print(games.columns)

print("\nReviews")

print(reviews.columns)

spark.stop()

print("\nExploration completed successfully.")


========== POSSIBLE JOIN KEYS ==========

Games
['appid', 'name', 'release_date', 'english', 'developer', 'publisher', 'platforms', 'required_age', 'categories', 'genres', 'steamspy_tags', 'achievements', 'positive_ratings', 'negative_ratings', 'average_playtime', 'median_playtime', 'owners', 'price']

Reviews
['app_id', 'app_name', 'review_text', 'review_score', 'review_votes']

Exploration completed successfully.
